In [ ]:
import pandas as pd
import numpy as np

# 1. Load all three datasets
print("Loading files...")
df_main = pd.read_csv('data/deliveries.csv')
df_2025 = pd.read_csv('data/ipl_2025_deliveries.csv')
df_2026 = pd.read_csv('data/ipl_2026_deliveries.csv')

print(f"Rows before merge -> Main: {len(df_main)}, 2025: {len(df_2025)}, 2026: {len(df_2026)}")

# 2. Define our standardization function
def standardize_new_data(df):
    df_new = df.copy()
    
    # Split the decimal 'over' (e.g., 14.2) into 'over' (14) and 'ball' (2)
    df_new['ball'] = (df_new['over'] * 10 % 10).round().astype(int)
    df_new['over'] = df_new['over'].astype(int)
    
    # Rename columns to match the 2008-2024 Kaggle standard
    rename_map = {
        'innings': 'inning',
        'striker': 'batter',
        'runs_of_bat': 'batsman_runs',
        'extras': 'extra_runs',
        'wicket_type': 'dismissal_kind'
    }
    df_new = df_new.rename(columns=rename_map)
    
    # Calculate missing aggregate columns
    df_new['total_runs'] = df_new['batsman_runs'] + df_new['extra_runs']
    df_new['is_wicket'] = df_new['player_dismissed'].notna().astype(int)
    
    # Reconstruct the 'extras_type' column based on the individual wide/noball tallies
    def get_extra_type(row):
        if row['wide'] > 0: return 'wides'
        if row['noballs'] > 0: return 'noballs'
        if row['legbyes'] > 0: return 'legbyes'
        if row['byes'] > 0: return 'byes'
        return np.nan
    df_new['extras_type'] = df_new.apply(get_extra_type, axis=1)
    
    # Add dummy 'non_striker' column since it is missing in the new data 
    # (It isn't used in our XGBoost Math Brain anyway)
    if 'non_striker' not in df_new.columns:
        df_new['non_striker'] = np.nan
        
    # Extract only the exact columns present in the main dataset
    expected_cols = df_main.columns.tolist()
    return df_new[expected_cols]

# 3. Apply the transformation to both new files
print("Standardizing 2025 and 2026 data...")
df_2025_clean = standardize_new_data(df_2025)
df_2026_clean = standardize_new_data(df_2026)

# 4. Stack them all together!
print("Merging all years into a single dataframe...")
df_combined = pd.concat([df_main, df_2025_clean, df_2026_clean], ignore_index=True)

# 5. Overwrite the main file with the master dataset
df_combined.to_csv('data/deliveries.csv', index=False)
print(f"✅ Success! Master deliveries.csv created with {len(df_combined)} total rows.")

 You don't need to look up the target_runs, winner, result_margin, or result. Your deliveries.csv file contains every single ball bowled!

If you sum up all the runs in Innings 1, you automatically get the target_runs.

If you compare Innings 1 total to Innings 2 total, you automatically know the winner and the result_margin.

In [ ]:
import pandas as pd

# Load both files
matches = pd.read_csv('../data/matches.csv')
deliveries = pd.read_csv('../data/deliveries.csv')

# Find the matches that are missing a 'winner'
missing_matches = matches[matches['winner'].isna()]['id'].tolist()

print(f"Calculating stats for {len(missing_matches)} matches...")

for match_id in missing_matches:
    match_balls = deliveries[deliveries['match_id'] == match_id]
    
    if not match_balls.empty:
        # Calculate Team Totals
        inn1_runs = match_balls[match_balls['inning'] == 1]['total_runs'].sum()
        inn2_runs = match_balls[match_balls['inning'] == 2]['total_runs'].sum()
        
        team1 = match_balls[match_balls['inning'] == 1]['batting_team'].iloc[0]
        # Handle cases where team2 might not have batted
        team2_balls = match_balls[match_balls['inning'] == 2]
        team2 = team2_balls['batting_team'].iloc[0] if not team2_balls.empty else "N/A"
        
        # Determine the Winner and Margin
        if inn1_runs > inn2_runs:
            matches.loc[matches['id'] == match_id, 'winner'] = team1
            matches.loc[matches['id'] == match_id, 'result_margin'] = inn1_runs - inn2_runs
            matches.loc[matches['id'] == match_id, 'result'] = 'runs'
        elif inn2_runs > inn1_runs:
            matches.loc[matches['id'] == match_id, 'winner'] = team2
            matches.loc[matches['id'] == match_id, 'result'] = 'wickets' # Margin requires wicket math, keeping it simple
        else:
            matches.loc[matches['id'] == match_id, 'winner'] = 'Tie'
            
        # Set Target
        matches.loc[matches['id'] == match_id, 'target_runs'] = inn1_runs + 1

# Save it back
matches.to_csv('../data/matches.csv', index=False)
print("✅ Calculated and updated winners, margins, and targets from raw delivery data!")

Calculating stats for 0 matches...
✅ Calculated and updated winners, margins, and targets from raw delivery data!


C:\Users\Ved Mungra\AppData\Local\Temp\ipykernel_31208\4168078178.py:5: DtypeWarning: Columns (0: non_striker) have mixed types. Specify dtype option on import or set low_memory=False.
  deliveries = pd.read_csv('../data/deliveries.csv')
